# Energy-based model and the road to diffusion models

When we trained our energy-based models, the training loop
called `train_model()` ran Persistent Contrastive Divergence (PCD), and we watched
the reconstruction error fall.
But why does that procedure approximate the log-likelihood
gradient? What is it actually computing? And is there a completely different route to the
same goal that avoids sampling altogether?

Let's attack these questions in stages to understand better the contributions from the partition function, the positive phase (term) and the negative phase.
- partition function as a explicit calculation, only possible in small model
- Gibbs sample and the negative phase, showing how the Markov Chain Monte Carlo gives the model expectation (negative phase), for CD-1, CD-$k$, and PCD
- the score function, a new and partition-function-free object that contains all the shape information of the distribution

For each step, we will use a very simple one-dimensional Gaussian mixture whose true distribution
is known analytically, so every approximation can be checked against the exact answer.

In [ ]:
from __future__ import print_function, division
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from scipy.special import expit as sigmoid   # numerically stable sigmoid
from scipy.stats import norm
import warnings
warnings.filterwarnings('ignore')

np.random.seed(42)
%matplotlib inline

# ── colour palette used throughout ──────────────────────────────────────────
C_DATA   = '#2979C5'   # blue  — data / truth
C_CD1    = '#E05C2A'   # orange — CD-1
C_CDK    = '#8B44AC'   # purple — CD-k
C_PCD    = '#2AAE6E'   # green  — PCD
C_SM     = '#C5275A'   # red    — score matching
C_DSM    = '#A07830'   # brown  — denoising SM

# Partition Function Calculation

## log-likelihood gradient

The energy-based model defines a probability distribution over visible
configurations $\mathbf{v}$:
$$p_\theta(\mathbf{v}) = \frac{1}{Z(\theta)} e^{-E_\theta(\mathbf{v})}, \qquad
Z(\theta) = \sum_{\mathbf{v}} e^{-E_\theta(\mathbf{v})}.$$

We identify $Z(\theta)$ as the partition function.

Taking the gradient of the average log-likelihood $L$ gives
$$\frac{\partial L}{\partial W_{i\mu}} =
\langle v_i h_\mu \rangle_\text{data}
- \langle v_i h_\mu \rangle_\theta.$$

The positive phase is easy: clamp $\mathbf{v}$ to a data point and compute
$\langle h_\mu \rangle = \sigma(\mathbf{v}^\top \mathbf{W} + \mathbf{b})$ analytically.

The negative phase is the problem. It requires averaging over all configurations
weighted by the current model distribution, basically computing an expectation value
under $p_\theta$.
For a binary RBM with $N_v$ visible and $N_h$ hidden units that
sum runs over $2^{N_v + N_h}$ terms.
For MNIST:
$$2^{784 + 200} = 2^{984} \approx 10^{296}.$$
For comparison, the observable universe contains roughly $10^{80}$ atoms, so it is completely intractable for us to count the configurations.

The cell below makes this concrete by computing the exact partition function for
tiny RBMs via brute force, and showing how the computing cost  scales with model complexity.

In [ ]:
# Exact partition function for a tiny binary Bernoulli-Bernoulli RBM
#
# For a Bernoulli-Bernoulli RBM:
#   E(v,h) = -a^T v  -  b^T h  -  v^T W h
# We enumerate all 2^(N_v + N_h) configurations and sum exp(-E).
# This is only feasible for very small N_v, N_h.

def exact_partition_function(W, a, b):
    """
    Enumerate all binary (v, h) configurations and return Z, together
    with the exact model expectations <v_i h_mu>.

    Args:
        W : (n_v, n_h) weight matrix
        a : (n_v,)     visible biases
        b : (n_h,)     hidden biases

    Returns:
        Z       : scalar partition function
        corr    : (n_v, n_h) exact model correlation <v_i h_mu>
    """
    n_v, n_h = W.shape
    n_total  = n_v + n_h

    # enumerate all 2^(n_v+n_h) binary vectors
    configs = np.array(
        [[int(b) for b in format(k, f'0{n_total}b')] for k in range(2**n_total)],
        dtype=np.float64
    )   # shape (2^n_total, n_total)

    v_configs = configs[:, :n_v]   # (2^n_total, n_v)
    h_configs = configs[:, n_v:]   # (2^n_total, n_h)

    # energy of every configuration
    E = -(v_configs @ a) - (h_configs @ b) - np.einsum('bi,ij,bj->b', v_configs, W, h_configs)

    weights = np.exp(-E)           # unnormalised Boltzmann weights
    Z       = weights.sum()
    p       = weights / Z          # exact probabilities

    # exact model correlations <v_i h_mu>
    corr = np.einsum('b,bi,bj->ij', p, v_configs, h_configs)
    return Z, corr


# Compare exact Z with CD estimate for a 4v x 3h RBM
rng = np.random.default_rng(42)
n_v_tiny, n_h_tiny = 4, 3
W_tiny = rng.normal(0, 0.3, (n_v_tiny, n_h_tiny))
a_tiny = rng.normal(0, 0.1, n_v_tiny)
b_tiny = rng.normal(0, 0.1, n_h_tiny)

Z_exact, corr_exact = exact_partition_function(W_tiny, a_tiny, b_tiny)
print(f'Tiny RBM  ({n_v_tiny}v + {n_h_tiny}h):  2^{n_v_tiny+n_h_tiny} = {2**(n_v_tiny+n_h_tiny)} configurations')
print(f'Exact Z = {Z_exact:.4f}')
print(f'Exact <v_1 h_1> = {corr_exact[0,0]:.6f}')

# ── How fast does the configuration count grow? ──────────────────────────────
sizes   = np.arange(2, 31)
n_cfgs  = 2.0 ** sizes
age_universe_s = 4.3e17   # seconds
rate_per_s     = 1e9      # 1 billion configurations per second

fig, ax = plt.subplots(figsize=(7, 3.5))
ax.semilogy(sizes, n_cfgs, color=C_DATA, lw=2)
ax.axhline(age_universe_s * rate_per_s, color='gray', ls='--', lw=1.2,
           label=r'$10^9$ configs/s $\times$ age of universe')
ax.axvline(n_v_tiny + n_h_tiny, color=C_CD1, ls=':', lw=1.5,
           label=f'tiny RBM ({n_v_tiny}v+{n_h_tiny}h)')
ax.axvline(28, color=C_PCD, ls=':', lw=1.5, label='MNIST scale (schematic)')
ax.set_xlabel('Total units $N_v + N_h$')
ax.set_ylabel('Number of configurations $2^{N_v+N_h}$')
ax.set_title('The partition function sum grows exponentially')
ax.legend(fontsize=9)
plt.tight_layout()
plt.show()
print('\nFor MNIST (784v + 200h): 2^984 ≈ 10^{296} configurations. '
      'Exact computation is impossible.')

We actually calculated $Z$ from the energy of each configuration, even if we didn't plot it.

Now you try:
- modify `exact_partition_function` to also return the exact marginal
$p(\mathbf{v}) = \sum_{\mathbf{h}} p(\mathbf{v}, \mathbf{h})$ for each visible
configuration.
- plot the 16 values as a bar chart and verify they sum to 1.

Gibbs sampling: the heat-bath algorithm

If we cannot sum over all configurations explicitly, we can sample from $p_\theta$
using a Markov chain Monte Carlo.
The standard tool in the RBM literature is block Gibbs
sampling, which is identical to the heat-bath algorithm from computational
statistical mechanics:

- Fix $\mathbf{v}$, sample $\mathbf{h} \sim p(\mathbf{h}|\mathbf{v})$.
- Fix $\mathbf{h}$, sample $\mathbf{v} \sim p(\mathbf{v}|\mathbf{h})$.
- Repeat as needed to get the desired number of samples.

Because the RBM is bipartite (no intra-layer connections), both conditionals
factorize, as shown in lecture:
$$p(h_\mu = 1 \mid \mathbf{v}) = \sigma\!\left(\sum_i W_{i\mu} v_i + b_\mu\right), \qquad
p(v_i = 1 \mid \mathbf{h}) = \sigma\!\left(\sum_\mu W_{i\mu} h_\mu + a_i\right).$$

This means that both layers can be sampled in parallel. The operations are just one matrix multiply and a Bernoulli random draw. This is much more efficient than standard single-spin-flip Metropolis algorithm, which can take a long time..

The time-average of any observable along a
long-enough chain converges to its equilibrium expectation value:
$$\frac{1}{T} \sum_{t=1}^{T} v_i^{(t)} h_\mu^{(t)} \xrightarrow{T\to\infty}
\langle v_i h_\mu \rangle_{p_\theta},$$
which looks very familiar!

The next cell demonstrates this for our tiny RBM, comparing the chain time-average
to the exact value computed by brute force.

In [ ]:
# Gibbs sampling on the tiny RBM

def gibbs_step_rbm(v, W, a, b):
    """One block Gibbs step for a Bernoulli-Bernoulli RBM."""
    h = (np.random.rand(W.shape[1]) < sigmoid(v @ W + b)).astype(float)
    v = (np.random.rand(W.shape[0]) < sigmoid(h @ W.T + a)).astype(float)
    return v, h

# Run a long Gibbs chain and accumulate the running mean of v_0 * h_0
n_steps = 50_000
v = np.random.randint(0, 2, n_v_tiny).astype(float)
running_mean = np.zeros(n_steps)
v_states     = np.zeros((n_steps, n_v_tiny))

for t in range(n_steps):
    v, h = gibbs_step_rbm(v, W_tiny, a_tiny, b_tiny)
    running_mean[t] = v[0] * h[0]
    v_states[t]     = v

# Plot: chain trace + running average convergence
fig, axes = plt.subplots(1, 2, figsize=(11, 3.5))

# left: trace of v_0 (shows the chain mixing)
axes[0].plot(v_states[:500, 0], color=C_DATA, lw=0.8)
axes[0].set_xlabel('Gibbs step')
axes[0].set_ylabel('$v_1$ (binary)')
axes[0].set_yticks([0, 1])

# right: running mean vs exact value
cumulative = np.cumsum(running_mean) / (np.arange(n_steps) + 1)
axes[1].semilogx(np.arange(1, n_steps+1), cumulative,
                 color=C_CD1, lw=1.5, label='Chain time-average')
axes[1].axhline(corr_exact[0, 0], color=C_DATA, lw=1.5, ls='--',
                label=f'Exact $\\langle v_1 h_1 \\rangle = {corr_exact[0,0]:.4f}$')
axes[1].set_xlabel('Number of Gibbs steps $T$')
axes[1].set_ylabel('Running mean of $v_1 h_1$')
axes[1].set_title('Running average converges to exact model expectation')
axes[1].legend(fontsize=9)

plt.tight_layout()
plt.show()
print(f'Final chain estimate: {cumulative[-1]:.6f}')
print(f'Exact value:          {corr_exact[0,0]:.6f}')
print(f'Error:                {abs(cumulative[-1]-corr_exact[0,0]):.2e}')

## Contrastive Divergence

Running a Gibbs chain to convergence at every gradient step is too expensive, but there is a shortcut:

- CD-$k$: start the chain from the data $\mathbf{v}^{(0)} = \mathbf{v}_\text{data}$,
run exactly $k$ Gibbs steps, and use the endpoint $\mathbf{v}^{(k)}$ as a
substitute for a model sample in the negative phase.

Compare this to "Persistent CD" (PCD): instead of restarting from data, maintain a set of
*persistent* fantasy particles that are updated continuously across gradient
steps. The chain never resets.

Neither of these computes the true log-likelihood gradient. Specifically:

PCD is a better approximation to the true gradient, but only if the persistent chains mix fast enough to track the model as it changes.

The cell below trains a Gaussian-visible / Bernoulli-hidden RBM on samples
from a one-dimensional Gaussian mixture using all three methods (CD-1, CD-10,
PCD), and compares their learned energy functions against the true log-density.

In [ ]:
# 1D Gaussian mixture: our ground-truth distribution
#
# p_data(v) = 0.5 * N(v; -2, 0.7^2)  +  0.5 * N(v; +2, 0.7^2)
#
#  - The true log-density and score are available analytically.
#  - The two modes represent a simple broken-symmetry scenario.
#  - We can directly visualise the learned energy landscape.

MU1, MU2 = -2.0, +2.0
SIG_DATA  = 0.7
W_MIX     = 0.5

def log_p_data(v):
    """True log-density of the Gaussian mixture (up to a constant)."""
    p = W_MIX * norm.pdf(v, MU1, SIG_DATA) + W_MIX * norm.pdf(v, MU2, SIG_DATA)
    return np.log(p + 1e-300)

def score_data(v):
    """True score  d/dv  log p_data(v)"""
    p1   = norm.pdf(v, MU1, SIG_DATA)
    p2   = norm.pdf(v, MU2, SIG_DATA)
    dp1  = -(v - MU1) / SIG_DATA**2 * p1
    dp2  = -(v - MU2) / SIG_DATA**2 * p2
    ptot = W_MIX * p1 + W_MIX * p2
    return (W_MIX * dp1 + W_MIX * dp2) / (ptot + 1e-300)

def sample_data(n):
    """Draw n iid samples from the Gaussian mixture."""
    which = np.random.rand(n) < W_MIX
    return np.where(which,
                    np.random.normal(MU1, SIG_DATA, n),
                    np.random.normal(MU2, SIG_DATA, n))

# visualise the distribution
v_grid = np.linspace(-5, 5, 500)
samples_train = sample_data(3000)

fig, ax = plt.subplots(figsize=(7, 3))
ax.hist(samples_train, bins=60, density=True, alpha=0.35,
        color=C_DATA, label='samples')
ax.plot(v_grid, np.exp(log_p_data(v_grid)), color=C_DATA, lw=2,
        label='$p_\\mathrm{data}(v)$')
ax.set_xlabel('$v$')
ax.set_ylabel('density')
ax.set_title('Training distribution: two-mode Gaussian mixture')
ax.legend()
plt.tight_layout()
plt.show()
print(f'Training set: {len(samples_train)} samples')

In [ ]:
# Gaussian-visible / Bernoulli-hidden RBM (1D visible)
#
# Visible units: v in R  (Gaussian, fixed variance sigma_v^2)
# Hidden  units: h in {0,1}  (Bernoulli)
#
# Energy:
#   E(v,h) = (v - a)^2 / (2 sigma_v^2)  -  b^T h  -  (v / sigma_v^2) * W^T h
#
# After marginalizing h, the free energy is:
#   F(v) = (v-a)^2/(2 sigma_v^2)  -  sum_mu log(1 + exp(b_mu + v*W_mu/sigma_v^2))
# so  log p_theta(v) = -F(v) - log Z_theta.
#
# The score (d/dv log p_theta) is:
#   s_theta(v) = -(v-a)/sigma_v^2  +  sum_mu  W_mu/sigma_v^2  * sigmoid(b_mu + v*W_mu/sigma_v^2)

class GaussianBernoulliRBM1D:
    """
    1D Gaussian-visible / Bernoulli-hidden RBM.

    Parameters
    ----------
    n_hidden  : int   — number of hidden units
    sigma_v   : float — fixed standard deviation of visible units
    lr        : float — learning rate
    """

    def __init__(self, n_hidden=8, sigma_v=1.0, lr=1e-3):
        self.n_h     = n_hidden
        self.sv      = sigma_v
        self.lr      = lr
        # parameters: visible bias a (scalar), hidden biases b, weights W
        self.a = 0.0
        self.b = np.zeros(n_hidden)
        self.W = np.random.normal(0, 0.1, n_hidden)   # shape (n_h,)

    # ── conditional distributions ────────────────────────────────────────────
    def prob_h_given_v(self, v):
        """P(h_mu=1|v) for each hidden unit. v: scalar or (batch,)."""
        # pre-activation: b_mu + v * W_mu / sigma_v^2
        return sigmoid(self.b + np.outer(v, self.W) / self.sv**2)  # (batch, n_h)

    def mean_v_given_h(self, h):
        """Mean of p(v|h): a + sigma_v^2 * W^T h. h: (batch, n_h)."""
        return self.a + h @ self.W   # (batch,)

    def sample_h(self, v):
        p = self.prob_h_given_v(np.atleast_1d(v))   # (batch, n_h)
        return (np.random.rand(*p.shape) < p).astype(float)

    def sample_v(self, h):
        mu = self.mean_v_given_h(np.atleast_2d(h))  # (batch,)
        return np.random.normal(mu, self.sv)

    # ── score function  d/dv log p_theta(v) ─────────────────────────────────
    def score(self, v):
        v = np.atleast_1d(v)
        mean_h = self.prob_h_given_v(v)              # (batch, n_h)
        return (-(v - self.a) / self.sv**2
                + mean_h @ self.W / self.sv**2)      # (batch,)

    # ── free energy (for computing unnormalised log-density) ─────────────────
    def free_energy(self, v):
        v = np.atleast_1d(v)
        fe = (v - self.a)**2 / (2 * self.sv**2)
        fe -= np.sum(np.log1p(np.exp(
            self.b + np.outer(v, self.W) / self.sv**2
        )), axis=1)
        return fe   # (batch,);  log p_theta(v) = -fe - log Z

    # ── one gradient-descent step ────────────────────────────────────────────
    def update(self, v_pos, v_neg):
        """
        Perform one CD or PCD parameter update.

        v_pos : (batch,) — data samples  (positive phase)
        v_neg : (batch,) — fantasy particles  (negative phase)
        """
        h_pos = self.prob_h_given_v(v_pos)   # mean-field  (batch, n_h)
        h_neg = self.prob_h_given_v(v_neg)

        # gradients: positive - negative
        grad_a = ((v_pos - self.a) / self.sv**2
                  - (v_neg - self.a) / self.sv**2).mean()
        grad_b = (h_pos - h_neg).mean(axis=0)
        grad_W = ((np.outer(v_pos, h_pos.mean(axis=0))
                 - np.outer(v_neg, h_neg.mean(axis=0)))
                 / len(v_pos)).sum(axis=0) / self.sv**2

        self.a += self.lr * grad_a
        self.b += self.lr * grad_b
        self.W += self.lr * grad_W


Now let's test all 3 of these for comparison.

In [ ]:
# Train three copies of the RBM: CD-1, CD-10, PCD

def train_rbm(method='cd1', k=1, n_epochs=300, batch_size=128,
              n_hidden=8, lr=5e-3, sigma_v=1.5, seed=0):
    """
    Train a GaussianBernoulliRBM1D and record reconstruction error per epoch.

    method : 'cd1', 'cdk', or 'pcd'
    k      : number of Gibbs steps (used for 'cdk' and 'pcd')
    """
    np.random.seed(seed)
    rbm = GaussianBernoulliRBM1D(n_hidden=n_hidden, sigma_v=sigma_v, lr=lr)
    # initialise visible bias to data mean
    rbm.a = samples_train.mean()

    n_train   = len(samples_train)
    val_data  = sample_data(1000)
    errors    = []

    # persistent chains (only used by PCD)
    fantasy_v = np.random.normal(0, 1, batch_size)

    for epoch in range(n_epochs):
        idx = np.random.permutation(n_train)

        for start in range(0, n_train - batch_size + 1, batch_size):
            v_pos = samples_train[idx[start:start + batch_size]]

            if method in ('cd1', 'cdk'):
                # restart chain from data
                vk = v_pos.copy()
                steps = 1 if method == 'cd1' else k
                for _ in range(steps):
                    hk = rbm.sample_h(vk)
                    vk = rbm.sample_v(hk)
                v_neg = vk

            else:   # pcd: continue persistent chains
                for _ in range(k):
                    hk = rbm.sample_h(fantasy_v)
                    fantasy_v = rbm.sample_v(hk)
                v_neg = fantasy_v.copy()

            rbm.update(v_pos, v_neg)

        # reconstruction error on validation set
        h_val  = rbm.sample_h(val_data)
        v_recon = rbm.mean_v_given_h(h_val)
        errors.append(np.mean((val_data - v_recon)**2))

    return rbm, np.array(errors)


print('Training CD-1 ...')
rbm_cd1,  err_cd1  = train_rbm(method='cd1',  k=1,  n_epochs=400, seed=0)
print('Training CD-10 ...')
rbm_cdk,  err_cdk  = train_rbm(method='cdk',  k=10, n_epochs=400, seed=1)
print('Training PCD-10 ...')
rbm_pcd,  err_pcd  = train_rbm(method='pcd',  k=10, n_epochs=400, seed=2)
print('Done.')

In [ ]:
import matplotlib.pyplot as plt

# Plot reconstruction error curves
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

for label, err, col in [('CD-1',  err_cd1,  C_CD1),
                         ('CD-10', err_cdk,  C_CDK),
                         ('PCD-10',err_pcd,  C_PCD)]:
    axes[0].plot(err, lw=1.5, color=col, label=label)
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Reconstruction MSE')
axes[0].set_title('Training curves: CD-1 vs CD-10 vs PCD-10')
axes[0].legend()

# Plot learned energy landscape vs true log-density
true_energy = -log_p_data(v_grid)
true_energy -= true_energy.min()   # shift for visual comparison

for label, rbm, col, ls in [('CD-1',  rbm_cd1,  C_CD1, '-'),
                         ('CD-10', rbm_cdk,  C_CDK, '--'),
                         ('PCD-10',rbm_pcd,  C_PCD, '-.')]:
    fe = rbm.free_energy(v_grid)
    fe -= fe.min()
    axes[1].plot(v_grid, fe, lw=1.5, color=col, label=label, ls=ls)

axes[1].plot(v_grid, true_energy, 'k--', lw=2, label='True $-\log p_\mathrm{data}$')
axes[1].set_xlabel('$v$')
axes[1].set_ylabel('Energy (shifted)')
axes[1].set_title('Learned energy landscapes')
axes[1].legend()

plt.tight_layout()
plt.show()

## Score Function: A Partition-Function-Free Object

### Magic $\log Z$ cancellation

The fundamental obstacle to computing the negative phase (negative term) is the intractable $Z$. But notice
what happens when we differentiate $\log p_\theta(\mathbf{v})$ with respect
to $\mathbf{v}$ rather than with respect to the parameters:

$$\mathbf{s}_\theta(\mathbf{v}) \;\equiv\;
\nabla_\mathbf{v} \log p_\theta(\mathbf{v})
= \nabla_\mathbf{v} \bigl[-E_\theta(\mathbf{v}) - \log Z(\theta)\bigr]
= -\nabla_\mathbf{v} E_\theta(\mathbf{v}).$$

$\log Z$ is a constant in $\mathbf{v}$, so it vanishes. The **score function**
$\mathbf{s}_\theta(\mathbf{v})$ is the negative gradient of the energy, and it
contains all the shape information of the distribution, but requires no
dependence on $Z$.


### Physics analogy

In classical mechanics, if $E(\mathbf{v})$ is a potential
energy, the $-\nabla_\mathbf{v} E$ is the force. The score function is exactly this force
field for the log-probability landscape. It points toward regions of higher
probability density, and its magnitude tells you how steep the density gradient
is.
For a Gaussian $p = N(\mu, \sigma^2)$ the score is
$s(v) = -(v-\mu)/\sigma^2$ — a linear restoring force, exactly as in a
harmonic oscillator.

For our Gaussian mixture, the true score is a sum of two such restoring forces,
weighted by the local density of each component. The cell below plots the
true score alongside the score estimated by the trained RBMs.

In [ ]:
# Visualise score functions example
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# left: density and score on the same axes (scaled)
s_true = score_data(v_grid)
ax = axes[0]
ax.fill_between(v_grid, 0, np.exp(log_p_data(v_grid)),
                alpha=0.18, color=C_DATA)
ax.plot(v_grid, np.exp(log_p_data(v_grid)), color=C_DATA, lw=2,
        label='$p_\\mathrm{data}(v)$')
ax2 = ax.twinx()
ax2.plot(v_grid, s_true, color='black', lw=1.5, ls='--',
         label='$s(v) = \\partial_v \\log p$')
ax2.axhline(0, color='gray', lw=0.8)
ax2.set_ylabel('Score $s(v)$', color='black')
ax.set_xlabel('$v$')
ax.set_title('Density and score of the Gaussian mixture')
lines1, lab1 = ax.get_legend_handles_labels()
lines2, lab2 = ax2.get_legend_handles_labels()
ax.legend(lines1+lines2, lab1+lab2, fontsize=9, loc='upper right')

# right: true score vs RBM scores
ax = axes[1]
ax.plot(v_grid, s_true, 'k--', lw=2, label='True score')
for label, rbm, col in [('CD-1',   rbm_cd1, C_CD1),
                         ('CD-10',  rbm_cdk, C_CDK),
                         ('PCD-10', rbm_pcd, C_PCD)]:
    ax.plot(v_grid, rbm.score(v_grid), color=col, lw=1.5, label=label)
ax.axhline(0, color='gray', lw=0.8)
ax.set_xlabel('$v$')
ax.set_ylabel('Score $s_\\theta(v)$')
ax.set_title('True score vs scores learned by CD variants')
ax.legend(fontsize=9)
ax.set_ylim(-8, 8)

plt.tight_layout()
plt.show()

### Interpretation of the score function

The score is zero at each mode and at the midpoint.

- positive score = pushed rightward
- negative score = pushed leftward.

The score at $v=0$ tells you whether the valley between modes is symmetric.

## Score Matching

If the score function is partition-function-free, can we build a training
objective $J$ around it? Yes!
The idea is to minimize the expected squared difference between the model
score and the data score:

$$J_\text{SM}(\theta) =
\frac{1}{2} \langle
\|\mathbf{s}_\theta(\mathbf{v}) - \nabla_\mathbf{v} \log p_\text{data}(\mathbf{v})\|^2 \rangle.$$

The problem is that $\nabla_\mathbf{v} \log p_\text{data}$ is unknown.
Hyvärinen's key insight was to show that

$$J_\text{SM}(\theta) =
\mathbb{E}_{p_\text{data}}\left[
\sum_i \partial_{v_i} s_{\theta,i}(\mathbf{v})
+\frac{1}{2}\|\mathbf{s}_\theta(\mathbf{v})\|^2
\right] + \text{const}$$

where the constant does not depend on $\theta$. This form involves only:
- $\mathbf{s}_\theta(\mathbf{v})$, the model score evaluated on data samples,
- $\partial_{v_i} s_{\theta,i}(\mathbf{v})$, the Jacobian trace of the score,
  i.e., how fast the score changes as $v_i$ changes.

Both quantities can be computed exactly from the model without any sampling.

### Example score matching for the 1D RBM

In 1D the Hyvärinen objective simplifies to:
$$J_\text{SM}(\theta) =
\mathbb{E}
\langle\partial_v s_\theta(v) + \tfrac{1}{2} s_\theta(v)^2\rangle.$$

For our Gaussian-Bernoulli RBM the score is:
$$s_\theta(v) = -\frac{v - a}{\sigma_v^2}
+\frac{1}{\sigma_v^2}\sum_\mu W_\mu \,\sigma(b_\mu + v W_\mu / \sigma_v^2)$$

and its derivative with respect to $v$ is:
$$\partial_v s_\theta(v) = -\frac{1}{\sigma_v^2}
+\frac{1}{\sigma_v^4}\sum_\mu W_\mu^2 \,
\sigma(b_\mu + v W_\mu/\sigma_v^2)\bigl[1 - \sigma(b_\mu + v W_\mu/\sigma_v^2)\bigr].$$

In [ ]:
# Score matching on the 1D Gaussian-Bernoulli RBM

def score_derivative(rbm, v):
    """
    Compute  d/dv s_theta(v)  for a GaussianBernoulliRBM1D.
    Uses the analytic formula derived above.

    Returns: array of shape (len(v),)
    """
    v    = np.atleast_1d(v)
    ph   = rbm.prob_h_given_v(v)               # (batch, n_h)
    # d/dv s = -1/sv^2  + (1/sv^4) * sum_mu W_mu^2 * ph_mu*(1-ph_mu)
    return -1.0 / rbm.sv**2 + (ph * (1.0 - ph) * rbm.W**2).sum(axis=1) / rbm.sv**4


def sm_loss(rbm, v_batch):
    """
    Hyvärinen score-matching loss on a batch of data.
    J_SM = E[ d/dv s(v)  +  0.5 * s(v)^2 ]
    """
    s  = rbm.score(v_batch)              # (batch,)
    ds = score_derivative(rbm, v_batch)  # (batch,)
    return (ds + 0.5 * s**2).mean()


def sm_gradients(rbm, v_batch):
    """
    Analytic gradients of the SM loss w.r.t. (a, b, W).
    Derived by differentiating J_SM through the score and its derivative.

    Returns: (grad_a, grad_b, grad_W)
    """
    v    = np.atleast_1d(v_batch)        # (B,)
    sv2  = rbm.sv**2
    ph   = rbm.prob_h_given_v(v)         # (B, n_h)
    qh   = ph * (1.0 - ph)              # (B, n_h)  — sigmoid derivative
    rh   = qh * (1.0 - 2.0 * ph)       # (B, n_h)  — second derivative factor
    s    = rbm.score(v)                  # (B,)
    ds   = score_derivative(rbm, v)      # (B,)

    # gradient of  J = ds + 0.5 * s^2  w.r.t. a:
    #   d(ds)/da = 0,   d(0.5 s^2)/da = s * ds/da = s * (1/sv^2)
    grad_a = (s / sv2).mean()

    # gradient w.r.t. b_mu:
    #   d(ds)/db_mu = W_mu^2/sv^4 * rh_mu
    #   d(0.5 s^2)/db_mu = s * (W_mu/sv^2) * qh_mu
    grad_b = ((rbm.W**2 / sv2**2) * rh + s[:, None] * (rbm.W / sv2) * qh).mean(axis=0)

    # gradient w.r.t. W_mu:
    #   d(ds)/dW_mu = (2*W_mu/sv^4)*qh_mu + (W_mu/sv^4)^2 * rh_mu * (v-a)*sv^2  [chain rule]
    #   Actually simpler to compute numerically via finite differences for the
    #   hidden terms; here we use a clean analytic form.
    #
    # ds = -1/sv^2  +  sum_mu W_mu^2 * qh_mu / sv^4
    # d(ds)/dW_mu = 2*W_mu*qh_mu/sv^4  +  W_mu^2*rh_mu*(v/sv^2)/sv^4  (chain rule thru ph)
    #            = (1/sv^4) * W_mu * (2*qh_mu + W_mu*rh_mu*v/sv^2)
    d_ds_dW = (2.0 * rbm.W * qh
               + rbm.W**2 * rh * (v[:, None] / sv2)) / sv2**2
    # d(0.5 s^2)/dW_mu = s * d(s)/dW_mu = s * (ph_mu + W_mu*qh_mu*v/sv^2) / sv^2
    d_s2_dW = s[:, None] * (ph + rbm.W * qh * v[:, None] / sv2) / sv2
    grad_W  = (d_ds_dW + d_s2_dW).mean(axis=0)

    return grad_a, grad_b, grad_W


def train_rbm_sm(n_epochs=400, batch_size=128, n_hidden=8, lr=2e-3,
                 sigma_v=1.5, seed=0):
    """Train a GaussianBernoulliRBM1D using score matching."""
    np.random.seed(seed)
    rbm   = GaussianBernoulliRBM1D(n_hidden=n_hidden, sigma_v=sigma_v, lr=lr)
    rbm.a = samples_train.mean()
    n_train = len(samples_train)
    losses  = []

    for epoch in range(n_epochs):
        idx = np.random.permutation(n_train)
        for start in range(0, n_train - batch_size + 1, batch_size):
            v_batch = samples_train[idx[start:start + batch_size]]
            ga, gb, gW = sm_gradients(rbm, v_batch)
            # ascent (maximise log-likelihood equivalent)
            rbm.a -= lr * ga
            rbm.b -= lr * gb
            rbm.W -= lr * gW

        losses.append(sm_loss(rbm, sample_data(500)))

    return rbm, np.array(losses)


print('Training RBM with score matching ...')
rbm_sm, losses_sm = train_rbm_sm(n_epochs=400)
print(f'Done. Final SM loss: {losses_sm[-1]:.4f}')

In [ ]:
# Compare learned energy: SM vs CD variants
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# left: SM training loss curve
axes[0].plot(losses_sm, color=C_SM, lw=1.5)
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Score-matching loss $\\mathcal{J}_\\mathrm{SM}$')
axes[0].set_title('Score matching training loss')

# right: energy landscape comparison
true_E = -log_p_data(v_grid)
true_E -= true_E.min()

for label, rbm, col, ls in [
        ('PCD-10', rbm_pcd, C_PCD, '-'),
        ('SM',     rbm_sm,  C_SM,  '-')]:
    fe = rbm.free_energy(v_grid)
    fe -= fe.min()
    axes[1].plot(v_grid, fe, color=col, lw=2, ls=ls, label=label)

axes[1].plot(v_grid, true_E, 'k--', lw=2, label='True $-\\log p_\\mathrm{data}$')
axes[1].set_xlabel('$v$')
axes[1].set_ylabel('Free energy (shifted)')
axes[1].set_title('Score matching vs PCD energy landscape')
axes[1].legend()

plt.tight_layout()
plt.show()

We can see that score matching required NO sampling at all, but it
converges to the same basic energy landscape as PCD via a completely different route.